# `signal_gate5.py` — Playground

Manual verification notebook for **Gate 5: Edge Check + EV** (rules only — zero Claude cost).

| Function | Status | Notes |
|---|---|---|
| `decide_gate5_signal(candidate, gate_results)` | ✅ built | Deterministic BUY/SKIP via `apply_edge_rules()` |

**Output:** `{passed, decision, win_probability, expected_value, edge, position_confidence, reason, trade_levels, gate_summary}`

Gate 5 runs **after** Gates 1–4 pass. Pattern mirrors Gate 3: Claude (Gates 2–4) produces structured fields → **pure rules** in `helpers/logic/ev_rules.py` make the final math decision. No API keys required.

| Setup | Typical result |
|---|---|
| score=3, BULLISH conf≥8, no caution | **BUY** — high EV |
| score=2, BULLISH conf=6, caution=True | **SKIP** — EV below threshold |
| gate3 not passed / missing price | **SKIP** — structured early return, no crash |

In [1]:
import sys
import pathlib

gate5_dir = pathlib.Path('.').resolve()
if not (gate5_dir / 'signal_gate5.py').exists():
    gate5_dir = pathlib.Path('backend/02_intelligence/gate5_signal').resolve()

intelligence_dir = gate5_dir.parent                          # → backend/02_intelligence
scanner_dir      = intelligence_dir.parent / '01_scanner'    # → backend/01_scanner

# gate3/gate4 playgrounds import upstream modules by bare name — add their dirs too.
upstream_dirs = [
    scanner_dir,
    intelligence_dir / 'gate1_hard_threat',
    intelligence_dir / 'gate2_news_threat',
    intelligence_dir / 'gate3_sentiment',
    intelligence_dir / 'gate4_contradiction',
]
for p in [str(intelligence_dir), str(gate5_dir), *map(str, upstream_dirs)]:
    if p not in sys.path:
        sys.path.insert(0, p)

from signal_gate5 import decide_gate5_signal

# Upstream chain — used only by the "Real scenario" section below (live API + Claude).
import momentum_scanner
from momentum_scanner import run_scan
from hard_threat_gate1 import get_shared_market_data, screen_gate1_hard_threats
from news_threat_gate2 import assess_gate2_news_threat
from sentiment_gate3 import evaluate_gate3_sentiment
from contradiction_gate4 import detect_gate4_contradiction
from helpers.fetchers.news import fetch_news
from helpers.fetchers.market import get_market_context

---
## Happy path — strong signals → BUY

Score 3 momentum, BULLISH conf 9, no caution. Expect `decision=BUY`, high EV, `position_confidence=HIGH`.

In [2]:
strong_candidate = {
    'ticker': 'NVDA',
    'price': 875.50,
    'atr': 12.30,
    'score': 3,
}
strong_gates = {
    'gate1': {'passed': True},
    'gate2': {'passed': True},
    'gate3': {
        'passed': True,
        'direction': 'BULLISH',
        'confidence': 9,
        'caution': False,
        'key_reason': 'Strong earnings beat and raised guidance',
    },
    'gate4': {
        'passed': True,
        'action': 'PASS',
        'contradiction_type': 'none',
        'risk_level': 'NONE',
        'reason': 'NONE',
    },
}

decide_gate5_signal(strong_candidate, strong_gates)

[gate5] NVDA: BUY — EV 1.062 | win_prob=69% | HIGH


{'passed': True,
 'decision': 'BUY',
 'win_probability': 0.6875,
 'expected_value': 1.0625,
 'edge': 0.5312,
 'position_confidence': 'HIGH',
 'reason': 'EV 1.062 ≥ 4% min edge — score=3, BULLISH conf=9, win_prob=69%',
 'trade_levels': {'entry': 875.5,
  'atr': 12.3,
  'stop': 857.05,
  'target': 912.4,
  'stop_pct': 0.0211,
  'target_pct': 0.0421,
  'reward_risk': 2.0},
 'gate_summary': 'Gate 1 (hard threats): PASS\nGate 2 (news threat): PASS — no catastrophic threat\nGate 3 (sentiment): BULLISH conf=9 — Strong earnings beat and raised guidance\nGate 4 (contradiction): PASS — none / NONE: NONE'}

---
## Variation — weak / marginal setup → SKIP

Score 2, minimum passing confidence, caution flag. Expect `decision=SKIP`, EV below 4%.

In [3]:
weak_candidate = {
    'ticker': 'NVDA',
    'price': 875.50,
    'atr': 12.30,
    'score': 2,
}
weak_gates = {
    'gate3': {
        'passed': True,
        'direction': 'BULLISH',
        'confidence': 6,
        'caution': True,
        'key_reason': 'Mixed headlines from medium sources',
    },
}

decide_gate5_signal(weak_candidate, weak_gates)

[gate5] NVDA: SKIP — EV -0.040 | win_prob=32%


{'passed': False,
 'decision': 'SKIP',
 'win_probability': 0.32,
 'expected_value': -0.04,
 'edge': -0.02,
 'position_confidence': 'LOW',
 'reason': 'EV -0.040 below 4% min edge — score=2, BULLISH conf=6, win_prob=32%',
 'trade_levels': {'entry': 875.5,
  'atr': 12.3,
  'stop': 857.05,
  'target': 912.4,
  'stop_pct': 0.0211,
  'target_pct': 0.0421,
  'reward_risk': 2.0},
 'gate_summary': 'Gate 3 (sentiment): BULLISH conf=6 (caution) — Mixed headlines from medium sources'}

---
## Side-by-side — compare strong vs weak on same ticker

In [4]:
for label, cand, gates in [
    ('strong', strong_candidate, strong_gates),
    ('weak',   weak_candidate,   weak_gates),
]:
    r = decide_gate5_signal(cand, gates)
    print(f"{label:6} {r['decision']:<4} EV={r['expected_value']:.3f} "
          f"win_prob={r['win_probability']:.0%} conf={r['position_confidence']}")

[gate5] NVDA: BUY — EV 1.062 | win_prob=69% | HIGH
strong BUY  EV=1.062 win_prob=69% conf=HIGH
[gate5] NVDA: SKIP — EV -0.040 | win_prob=32%
weak   SKIP EV=-0.040 win_prob=32% conf=LOW


---
## Failure path — gate3 not passed / missing price

Structured SKIP (same return shape as a normal run) — mirrors Gate 3 blocking on empty headlines.

In [5]:
decide_gate5_signal({'ticker': 'NVDA', 'price': 875.50, 'atr': 12.30}, {})

decide_gate5_signal({'ticker': 'BAD'}, weak_gates)

[gate5] NVDA: SKIP — gate3_not_passed
[gate5] BAD: SKIP — missing_price_or_atr


{'passed': False,
 'decision': 'SKIP',
 'win_probability': 0.0,
 'expected_value': 0.0,
 'edge': 0.0,
 'position_confidence': 'LOW',
 'reason': 'missing_price_or_atr',
 'trade_levels': {},
 'gate_summary': 'Gate 3 (sentiment): BULLISH conf=6 (caution) — Mixed headlines from medium sources'}

## Real scenario — full pipeline (live)

The genuine end-to-end path, not synthetic dicts: live `run_scan()` → **Gate 1** (market data) → live `fetch_news` → **Gate 2 & 3** (Claude) → live `get_market_context` → **Gate 4** (Claude) → **Gate 5** edge check. This is what the bot actually sees on a given day, so results vary with the news window and the live tape.

⚠️ **Live API + Claude calls** — roughly 3 Claude calls per surviving candidate. Needs `.env` keys: `ANTHROPIC_API_KEY` plus a news source (`ALPACA_API_KEY` + `ALPACA_SECRET_KEY`, or `NEWSAPI_KEY`). Mirrors the live sections in the gate3 / gate4 playgrounds.

**`sector`:** Gate 1 and Gate 4 both require `candidate['sector']`. `run_scan()` now includes the `sector` column directly, so no extra lookup is needed. Candidates with no sector are skipped rather than guessed.

In [ ]:
import pandas as pd

# --- run-level inputs (Gate 1 portfolio checks) — tweak freely ---
portfolio_value = 100_000.0
daily_pnl       = 0.0
TOP_N           = 10          # cap surviving candidates to bound Claude cost

# run_scan() reads a cwd-relative path — anchor it to the real watchlist regardless of kernel cwd.
momentum_scanner.WATCHLIST_PATH = str(scanner_dir / 'data' / 'watchlist.csv')


def _row(ticker, decision, g5=None):
    """One results-table row; EV/win_prob/confidence only when Gate 5 actually ran."""
    return {
        'ticker': ticker,
        'final_decision': decision,
        'ev': round(g5['expected_value'], 3) if g5 else None,
        'win_prob': round(g5['win_probability'], 3) if g5 else None,
        'position_confidence': g5['position_confidence'] if g5 else None,
    }


candidates_df = run_scan(min_score=2)
if candidates_df is None or candidates_df.empty:
    print('[real] no candidates from run_scan — check watchlist.csv')
else:
    shared = get_shared_market_data()        # market-wide data, fetched once for all Gate 1 calls
    rows, processed = [], 0

    for _, r in candidates_df.iterrows():
        if processed >= TOP_N:
            break
        ticker = str(r['ticker'])
        sector = r['sector']
        if pd.isna(sector):
            print(f'{ticker:<5} skipped — no sector in watchlist')
            continue
        processed += 1

        candidate = {
            'ticker': ticker,
            'sector': sector,
            'price': float(r['price']),
            'atr': float(r['atr']),
            'score': int(r['score']),
        }

        # Gate 1 — hard threats
        g1 = screen_gate1_hard_threats(candidate, shared, portfolio_value, daily_pnl)
        if not g1['passed']:
            rows.append(_row(ticker, f"BLOCKED_G1:{g1.get('block_reason')}"))
            continue

        # Gates 2 & 3 share a single news fetch
        headlines = fetch_news(ticker) or []
        g2 = assess_gate2_news_threat(candidate, headlines)
        if not g2['passed']:
            rows.append(_row(ticker, 'BLOCKED_G2'))
            continue
        g3 = evaluate_gate3_sentiment(candidate, headlines)
        if not g3['passed']:
            rows.append(_row(ticker, 'BLOCKED_G3'))
            continue

        # Gate 4 — contradiction vs the live market backdrop
        market_context = get_market_context(sector)
        if market_context is None:
            rows.append(_row(ticker, 'BLOCKED_G4:no_market_context'))
            continue
        g4 = detect_gate4_contradiction(candidate, g3, market_context)
        if not g4['passed']:
            rows.append(_row(ticker, f"BLOCKED_G4:{g4.get('action')}"))
            continue

        # Gate 5 — edge check + EV
        g5 = decide_gate5_signal(candidate, {'gate1': g1, 'gate2': g2, 'gate3': g3, 'gate4': g4})
        rows.append(_row(ticker, g5['decision'], g5))

    results_df = pd.DataFrame(rows)
    print('\n' + results_df.to_string(index=False))
    buys = int((results_df['final_decision'] == 'BUY').sum()) if not results_df.empty else 0
    print(f'\nBUY signals: {buys} / {len(results_df)} processed')

---
## Free-play